In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

In [ ]:
bp_all = pd.read_csv(
    '<PATH_TO_SINGLE_AGE_PRE_POST_TWOYEARS_ON_DRUGS_TSV>',
    sep = '\t')

In [ ]:
drug_classes = [
    'ACE_inhibitor',
    'angiotensin_receptor_blocker',
    'calcium_channel_blocker',
    'beta_blocker',
    'diuretic',
    'statin'
]

results = []
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for i, drug in enumerate(drug_classes):
    sub = bp_all[bp_all[drug] == 1]
    sbp_stat, sbp_p = ttest_ind(sub['SBP_pre'].dropna(), sub['SBP_post'].dropna(), equal_var=True)
    dbp_stat, dbp_p = ttest_ind(sub['DBP_pre'].dropna(), sub['DBP_post'].dropna(), equal_var=True)
    results.append((drug, sbp_stat, sbp_p, dbp_stat, dbp_p))

    df_melted = sub[['SBP_pre','SBP_post','DBP_pre','DBP_post']].melt(
        var_name='Measure', value_name='Value'
    )
    sns.boxplot(x='measure',
                y='Value',
                data=df_melted,
                order=['SBP_pre','SBP_post','DBP_pre','DBP_post'],
                palette=["#9DB4A5","#5B7D8D","#9DB4A5","#5B7D8D"],
                showfliers=False,
                ax=axes[i])
    axes[i].set_title(drug)

plt.tight_layout()
plt.savefig(output_png, dpi=150, bbox_inches="tight")
plt.show()

for drug, sbp_stat, sbp_p, dbp_stat, dbp_p in results:
    print(f"{drug}: SBP t={sbp_stat:.3g}, p={sbp_p:.3g}; DBP t={dbp_stat:.3g}, p={dbp_p:.3g}")

In [ ]:
bp_all = pd.read_csv(
    '<PATH_TO_SINGLE_AGE_PRE_POST_TWOYEARS_TSV>',
    sep = '\t')

In [ ]:
drug_cols = [
    'ACE_inhibitor',
    'angiotensin_receptor_blocker',
    'calcium_channel_blocker',
    'beta_blocker',
    'diuretic',
    'statin'
]
bp_all['any_drug'] = (bp_all[drug_cols].sum(axis=1) > 0).map({False:'no drugs', True)

group_no_drug = bp_all[bp_all['any_drug'] == 'no drugs']
group_on_drug = bp_all[bp_all['any_drug'] == 'on drugs']
sbp_t, sbp_p = ttest_ind(
    group_no_drug['SBP_pre'].dropna(),
    group_on_drug['SBP_pre'].dropna(),
    equal_var=True
)
dbp_t, dbp_p = ttest_ind(
    group_no_drug['DBP_pre'].dropna(),
    group_on_drug['DBP_pre'].dropna(),
    equal_var=True
)
print(f"SBP_pre t-test: t={sbp_t:.3g}, p={sbp_p:.3g}")
print(f"DBP_pre t-test: t={dbp_t:.3g}, p={dbp_p:.3g}")

df_melt = bp_all.melt(
    id_vars='any_drug',
    value_vars=['SBP_pre','DBP_pre'],
    var_name='measure',
    value_name='BP [mmHg]'
)

plt.figure(figsize=(3,4))
sns.boxplot(
    x='measure',
    y='BP [mmHg]',
    hue='any_drug',
    data=df_melt,
    showfliers=False,
    palette=["#9DB4A5", "#5B7D8D"]
)

plt.tight_layout()
plt.savefig('on_of_drugs_bp', dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
SBP_snps_all = pd.read_csv(
    '<PATH_TO_TWOYEARS_SBP_MARKERS_TSV>',
    sep = '\t'
)

SBP_snps_on = pd.read_csv(
    '<PATH_TO_TWOYEARS_ON_DRUGS_SBP_MARKERS_TSV>',
    sep = '\t'
)

SBP_snps_5 = pd.read_csv(
    '<PATH_TO_PREPOST_SBP_MARKERS_TSV>',
    sep = '\t'
)

In [ ]:
SBP_snps_5[SBP_snps_5.phenotype == 'angiotensin_receptor_blocker_1'].to_csv('<PATH_TO_ARB_LOFS_FULL_CSV>')

In [ ]:
SBP_snps_on[SBP_snps_on.phenotype == 'angiotensin_receptor_blocker'].to_csv('<PATH_TO_ARB_SNP_CSV>')

In [ ]:
drug_snps = pd.concat([SBP_snps_all[(SBP_snps_all.rsID == 'rs186696265')],SBP_snps_on[(SBP_snps_on.phenotype == 'statin') | (SBP_snps_on.phenotype == 'angiotensin_receptor_blocker')]])

In [ ]:
drug_snps = list(drug_snps.rsID)

In [ ]:
drug_snps.remove('APOB')

In [ ]:
# data analysis for drug-taking SNPs and LoFs

In [ ]:
SBP_snps_all = pd.concat([SBP_snps_all[(SBP_snps_all.rsID == 'rs186696265')],SBP_snps_on[(SBP_snps_on.phenotype == 'statin') | (SBP_snps_on.phenotype == 'angiotensin_receptor_blocker')]])
to_drop = [
    'ARTN', 'SPRR2E', 'HOMER1', 'RAB24',
    'FOXL3', 'PAXIP1', 'ADGRA1', 'BORCS5', 'DMC1'
]

SBP_snps_5 = SBP_snps_5[~SBP_snps_5.rsID.isin(to_drop)]

In [ ]:
def stacked_lof_snp_bydrug(df_age_averaged, df_age_aware):
    
    def count_lof_snp(df, phenotype_value):
        sub = df[df["phenotype"] == phenotype_value]
        is_snp = sub["rsID"].str.lower().str.contains("rs", na=False)
        snp_count = is_snp.sum()
        lof_count = (~is_snp).sum()
        return lof_count, snp_count

    lof_statin_avg, snp_statin_avg = count_lof_snp(df_age_averaged, "statin")
    lof_arb_avg, snp_arb_avg = count_lof_snp(df_age_averaged, "angiotensin_receptor_blocker")

    lof_statin_aware, snp_statin_aware = count_lof_snp(df_age_aware, "statin_1")
    lof_arb_aware, snp_arb_aware = count_lof_snp(df_age_aware, "angiotensin_receptor_blocker_1")

    x_positions = [0, 1, 2, 3]
    x_labels = [
        "statins (all ages)", 
        "ARBs (all ages)",
        "statins (age <50)",
        "ARBs (age <50)"
    ]

    fig, ax = plt.subplots(figsize=(7.2, 5))

    ax.bar(
        x_positions[0],
        lof_statin_avg,
        color="#9DB4A5",
        label="LoF variants"
    )
    ax.bar(
        x_positions[0],
        snp_statin_avg,
        bottom=lof_statin_avg,
        color="#5B7D8D",
        label="SNPs"
    )

    ax.bar(
        x_positions[1],
        lof_arb_avg,
        color="#9DB4A5"
    )
    ax.bar(
        x_positions[1],
        snp_arb_avg,
        bottom=lof_arb_avg,
        color="#5B7D8D"
    )

    ax.bar(
        x_positions[2],
        lof_statin_aware,
        color="#9DB4A5"
    )
    ax.bar(
        x_positions[2],
        snp_statin_aware,
        bottom=lof_statin_aware,
        color="#5B7D8D"
    )

    ax.bar(
        x_positions[3],
        lof_arb_aware,
        color="#9DB4A5"
    )
    ax.bar(
        x_positions[3],
        snp_arb_aware,
        bottom=lof_arb_aware,
        color="#5B7D8D"
    )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, fontsize=12)
    ax.set_ylabel("number of variants")
    ax.tick_params(axis='y', labelsize=14)

    ax.legend(loc="best", fontsize=16)

    plt.tight_layout()
    plt.savefig('<PATH_TO_STACKED_SNP_LOF_PNG>', dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
df_age_averaged = SBP_snps_all[
    (SBP_snps_all["phenotype"] == "statin") |
    (SBP_snps_all["phenotype"] == "angiotensin_receptor_blocker")
]
df_age_aware = SBP_snps_5[
    (SBP_snps_5["phenotype"] == "statin_1") |
    (SBP_snps_5["phenotype"] == "angiotensin_receptor_blocker_1")
]
stacked_lof_snp_bydrug(df_age_averaged, df_age_aware)


In [ ]:
df_age_averaged = df_age_averaged[~(df_age_averaged['rsID'] == 'APOB')]

In [ ]:
ot = pd.read_csv('<PATH_TO_GWAS_OVERLAP_OPEN_TARGETS_WITH_DESCRIPTION_CSV>')
gc = pd.read_csv('<PATH_TO_GWAS_OVERLAP_TOP_GWAS_CSV>')

In [ ]:
ot2 = pd.read_csv('<PATH_TO_GWAS_OVERLAP_OPEN_TARGETS_CSV>')

In [ ]:
ot[ot['BASE_SNP'].isin(drug_snps)].to_csv('<PATH_TO_DRUG_SNPS_OPEN_TARGETS_CSV>')

In [ ]:
results = {}
target_rsids = df_age_averaged["rsID"].unique()
for rsid in target_rsids:
    df_filt1 = ot[ot["BASE_SNP"] == rsid].copy()
    df_filt2 = gc[gc["BASE_SNP"] == rsid].copy()
    results[rsid] = (df_filt1, df_filt2)

def check_statin(results):
    statin_map = {}
    for rsid, (df1, df2) in results.items():
        text1 = "not found"
        text2 = "not found"

        if not df1.empty and "TRAIT_REPORTED" in df1.columns:
            # Rows matching statin/hmg in TRAIT_REPORTED
            matches1 = df1.loc[
                df1["TRAIT_REPORTED"].str.contains("(statin|hmg)", case=False, na=False, regex=True)
            ]
            if not matches1.empty:
                collected = []
                for row in matches1.itertuples():
                    txt = row.TRAIT_REPORTED
                    # If DB_SOURCE column exists and contains FINNGEN => append "[FINNGEN]"
                    if hasattr(row, "DB_SOURCE") and row.DB_SOURCE and "FINNGEN" in row.DB_SOURCE:
                        txt += " [FINNGEN]"
                    collected.append(txt)
                if collected:
                    text1 = "; ".join(set(collected))

        if not df2.empty and "DISEASE/TRAIT" in df2.columns:
            # Rows matching statin/hmg in DISEASE/TRAIT
            matches2 = df2.loc[
                df2["DISEASE/TRAIT"].str.contains("(statin|hmg)", case=False, na=False, regex=True)
            ]
            if not matches2.empty:
                text2 = "; ".join(matches2["DISEASE/TRAIT"].unique())

        statin_map[rsid] = (text1, text2)

    return statin_map

rs12608822 is associated with cholesterol-lowering medication, it also replicates in finngen: https://r12.finngen.fi/variant/19:19730943-C-T

the rs186696265 also directly replicates in finngen [https://r12.finngen.fi/variant/6:160690668-C-T]

In [ ]:
x_positions = [0, 1]
x_labels = ["ARBs", "statins"]

novel_color = "#5B7D8D"
ukb_color   = "#d2f1ff"
finn_color  = "#82a3c9"

fig, ax = plt.subplots(figsize=(5,4))

ax.bar(x_positions[0], 1, color=novel_color, label="novel")
#ax.bar(x_positions[0], 0, bottom=1, color=ukb_color, label="replicated in UKB only")
ax.bar(x_positions[0], 0, bottom=1, color=finn_color, label="replicated in FINNGEN")

ax.bar(x_positions[1], 12, color=finn_color)

ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels, fontsize=12)
ax.set_ylabel("Number of SNPs", fontsize=12)

ax.legend(loc="upper left", fontsize=11)
fig.tight_layout()
plt.show()